# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | ** | 
| **Mehmet Kilinc 25135007** | *khaleelalaqel* |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [1]:
# Prompt 1: <CHAT-TITEL>
"""Imports en algemene instellingen voor het hele notebook."""
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import (GradientBoostingClassifier,
                              RandomForestClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             average_precision_score, confusion_matrix,
                             f1_score, make_scorer, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Vaste instellingen zodat alle resultaten reproduceerbaar zijn
RANDOM_STATE = 42
N_FOLDS = 5
N_JOBS = -1  # gebruik alle CPU-kernen
DATA_DIR = Path("data")
SUBMISSION_DIR = Path("submissions")
SUBMISSION_DIR.mkdir(exist_ok=True)

# Waarschuwingen (bijv. over convergentie tijdens het tunen) maken de
# output onoverzichtelijk; daarom zetten we ze uit.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)


<br>

### 2.2 Klasse `StrokeDataset`: data laden, controleren en opschonen
Deze klasse bundelt alles wat met de ruwe data te maken heeft. Door dit in één klasse te zetten, worden train- en testdata altijd op precies dezelfde manier bewerkt. Dat voorkomt fouten zoals een kolom die wel uit train maar niet uit test wordt verwijderd.

| Methode | Wat doet het? |
|---|---|
| `load()` | Leest `train.csv` en `test.csv` in. |
| `column_overview()` | Datatype, aantal unieke en ontbrekende waarden, minimum en maximum per kolom. |
| `check_sklearn_requirements()` | Controleert de minimale eisen van scikit-learn (paragraaf 3.5). |
| `readable_frame()` | Zet de dummy-kolommen terug naar leesbare categorieën, alleen voor de grafieken. |
| `clean()` | Schoont train en test op dezelfde manier op (paragraaf 3.7). |
| `split()` | Maakt een gestratificeerde train- en validatieset. |

In [2]:
# Prompt 1: <CHAT-TITEL>
class StrokeDataset:
    """Laadt, controleert en schoont de stroke-data van Kaggle.

    Parameters
    ----------
    data_dir : Path
        Map waarin train.csv en test.csv staan.
    """

    TARGET = "stroke"
    ID_COLUMN = "id"
    NUMERIC_COLUMNS = ["age", "avg_glucose_level", "bmi"]
    # Van iedere binaire variabele houden we één dummy-kolom over
    REDUNDANT_COLUMNS = ["gender_Female", "ever_married_No",
                         "Residence_type_Rural"]
    DUMMY_GROUPS = ["gender", "ever_married", "work_type",
                    "Residence_type", "smoking_status"]

    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.train_raw = None
        self.test_raw = None
        self.train = None
        self.test = None
        self.test_ids = None
        self.cleaning_log = []

    def load(self):
        """Leest train.csv en test.csv in als DataFrames."""
        self.train_raw = pd.read_csv(self.data_dir / "train.csv")
        self.test_raw = pd.read_csv(self.data_dir / "test.csv")
        return self

    def column_overview(self):
        """Geeft per kolom datatype, uniek, ontbrekend, min en max."""
        data = self.train_raw
        return pd.DataFrame({
            "datatype": data.dtypes.astype(str),
            "aantal uniek": data.nunique(),
            "ontbrekend": data.isna().sum(),
            "minimum": data.min(),
            "maximum": data.max(),
        })

    def check_sklearn_requirements(self):
        """Controleert de minimale eisen van scikit-learn voor X en y."""
        train, test = self.train_raw, self.test_raw
        features = train.columns.drop(self.TARGET)
        checks = {
            "Alle kolommen zijn numeriek of boolean": all(
                pd.api.types.is_numeric_dtype(train[column])
                for column in train.columns),
            "Geen ontbrekende waarden in train":
                int(train.isna().sum().sum()) == 0,
            "Geen ontbrekende waarden in test":
                int(test.isna().sum().sum()) == 0,
            "Geen dubbele patiënten (rijen zonder id)":
                not train.drop(columns=self.ID_COLUMN).duplicated().any(),
            "Zelfde feature-kolommen in train en test":
                list(features) == list(test.columns),
            "Doelvariabele is binair (0/1)":
                set(train[self.TARGET].unique()) == {0, 1},
        }
        return pd.DataFrame({"voldaan": list(checks.values())},
                            index=list(checks.keys()))

    def readable_frame(self):
        """Zet de dummy-kolommen terug naar categorieën (voor de EDA)."""
        data = self.train_raw.copy()
        for group in self.DUMMY_GROUPS:
            prefix = group + "_"
            columns = [col for col in data.columns if col.startswith(prefix)]
            labels = data[columns].idxmax(axis=1).str.replace(
                prefix, "", regex=False)
            # Rijen waarin alle dummies False zijn: categorie 'Unknown'
            labels[~data[columns].any(axis=1)] = "Unknown"
            data[group] = labels
            data = data.drop(columns=columns)
        return data

    def clean(self):
        """Schoont train en test op (motivatie: paragraaf 3.7)."""
        train = self.train_raw.copy()
        test = self.test_raw.copy()

        # 1. Rijen met gender 'Other' zijn te zeldzaam om van te leren
        other_rows = train["gender_Other"]
        train = train.loc[~other_rows]
        self._log(f"{other_rows.sum()} rijen met gender 'Other' "
                  "verwijderd uit train")

        # 2. Id bewaren voor de submission en daarna als feature weghalen
        self.test_ids = test[self.ID_COLUMN].copy()
        drop_columns = ([self.ID_COLUMN, "gender_Other"]
                        + self.REDUNDANT_COLUMNS)
        train = train.drop(columns=drop_columns)
        test = test.drop(columns=drop_columns)
        self._log("kolommen verwijderd: " + ", ".join(drop_columns))

        # 3. Booleans omzetten naar 0/1 zodat alle features getallen zijn
        bool_columns = train.select_dtypes(include="bool").columns
        train[bool_columns] = train[bool_columns].astype(int)
        test[bool_columns] = test[bool_columns].astype(int)
        self._log(f"{len(bool_columns)} boolean-kolommen omgezet naar 0/1")

        self.train = train.reset_index(drop=True)
        self.test = test.reset_index(drop=True)
        return self

    def _log(self, message):
        """Bewaart een regel tekst over een opschoonstap."""
        self.cleaning_log.append(message)

    def get_features_and_target(self):
        """Geeft de features X en doelvariabele y van de opgeschoonde train."""
        features = self.train.drop(columns=self.TARGET)
        target = self.train[self.TARGET]
        return features, target

    def split(self, test_size=0.2):
        """Maakt een gestratificeerde train- en validatieset."""
        features, target = self.get_features_and_target()
        return train_test_split(features, target, test_size=test_size,
                                stratify=target,
                                random_state=RANDOM_STATE)

<br>

### 2.3 Klasse `EdaPlotter`: visualisaties voor de EDA
Alle grafieken van de EDA worden door één klasse gemaakt. Daardoor hebben ze dezelfde opmaak en is de code van hoofdstuk 3 kort.

In [4]:
# Prompt 1: <CHAT-TITEL>
class EdaPlotter:
    """Maakt de visualisaties voor de exploratieve data-analyse.

    Parameters
    ----------
    data : pd.DataFrame
        Leesbare dataset (categorieën in plaats van dummy-kolommen).
    target : str
        Naam van de doelvariabele.
    """

    COLORS = {0: "#4C72B0", 1: "#C44E52"}

    def __init__(self, data, target="stroke"):
        self.data = data
        self.target = target
        self.overall_rate = data[target].mean() * 100

    def target_distribution(self):
        """Toont het aantal en percentage patiënten per klasse."""
        counts = self.data[self.target].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(["Geen stroke (0)", "Stroke (1)"], counts.values,
                      color=list(self.COLORS.values()))
        for bar, count in zip(bars, counts.values):
            share = count / counts.sum() * 100
            ax.annotate(f"{count:,} ({share:.1f}%)",
                        (bar.get_x() + bar.get_width() / 2,
                         bar.get_height()),
                        ha="center", va="bottom")
        ax.set_title("Verdeling van de doelvariabele stroke")
        ax.set_ylabel("Aantal patiënten")
        plt.tight_layout()
        plt.show()

    def numeric_distributions(self, columns):
        """Toont per numerieke feature een dichtheidsplot en boxplot."""
        fig, axes = plt.subplots(2, len(columns),
                                 figsize=(5.5 * len(columns), 8))
        for i, column in enumerate(columns):
            sns.kdeplot(data=self.data, x=column, hue=self.target,
                        common_norm=False, fill=True,
                        palette=self.COLORS, ax=axes[0, i])
            axes[0, i].set_title(f"Dichtheid van {column} per klasse")
            sns.boxplot(data=self.data, x=self.target, y=column,
                        hue=self.target, palette=self.COLORS,
                        legend=False, ax=axes[1, i])
            axes[1, i].set_title(f"Boxplot van {column} per klasse")
        plt.tight_layout()
        plt.show()

    def rate_per_category(self, columns):
        """Toont het stroke-percentage per categorie van iedere feature."""
        n_cols = 4
        n_rows = int(np.ceil(len(columns) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(19, 4.6 * n_rows))
        axes = axes.flatten()
        for ax, column in zip(axes, columns):
            summary = self.data.groupby(column)[self.target].agg(
                ["mean", "size"])
            ax.bar(summary.index.astype(str), summary["mean"] * 100,
                   color="#55A868")
            ax.axhline(self.overall_rate, color="red", linestyle="--",
                       label=f"gemiddeld ({self.overall_rate:.2f}%)")
            for x_pos, (rate, size) in enumerate(
                    zip(summary["mean"], summary["size"])):
                ax.annotate(f"n={size:,}", (x_pos, rate * 100),
                            ha="center", va="bottom", fontsize=8)
            ax.set_title(f"Stroke-% per {column}")
            ax.set_ylabel("Stroke (%)")
            ax.tick_params(axis="x", rotation=30)
        for ax in axes[len(columns):]:
            ax.set_visible(False)
        axes[0].legend(loc="upper left")
        plt.tight_layout()
        plt.show()

    def rate_per_age_group(self, bin_width=5):
        """Toont het stroke-percentage per leeftijdsgroep."""
        bins = np.arange(0, self.data["age"].max() + bin_width, bin_width)
        groups = pd.cut(self.data["age"], bins=bins, right=False)
        summary = self.data.groupby(groups, observed=True)[
            self.target].agg(["mean", "size"])
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot([interval.left for interval in summary.index],
                summary["mean"] * 100, marker="o", color="#C44E52")
        ax.set_title("Stroke-% per leeftijdsgroep van 5 jaar")
        ax.set_xlabel("Leeftijd (ondergrens van de groep)")
        ax.set_ylabel("Stroke (%)")
        plt.tight_layout()
        plt.show()

    def target_correlations(self, numeric_data):
        """Toont de correlatie van iedere feature met de target."""
        target_corr = (numeric_data.corr()[self.target]
                       .drop(self.target).sort_values())
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(target_corr.index, target_corr.values,
                color=np.where(target_corr > 0, "#C44E52", "#4C72B0"))
        ax.set(title="Correlatie (Pearson) van iedere feature met stroke",
               xlabel="Correlatie")
        plt.tight_layout()
        plt.show()

<br>

### 2.4 Evaluatiefuncties
Deze functies gebruiken we bij alle modellen:

- `get_scores()` geeft per patiënt een score: de kans op stroke (`predict_proba`) of, bij een SVM, de afstand tot het hypervlak (`decision_function`).
- `find_best_threshold()` zoekt de **beslisdrempel** met de hoogste F1-score. Een model voorspelt `stroke = 1` als de score boven die drempel ligt. Waarom dit belangrijk is, staat in paragraaf 5.1.
- `F1_BEST_SCORER` is de score die `GridSearchCV` gebruikt: de F1-score bij de beste drempel.
- `metrics_from_confusion()` rekent de metrics **zelf** uit met de formules uit paragraaf 4.2, zodat we kunnen laten zien hoe ze worden berekend.

In [3]:
# Prompt 1: <CHAT-TITEL>
def get_scores(model, features):
    """Geeft per patiënt een score: kans op stroke of decision function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def f1_per_threshold(y_true, scores):
    """Berekent de F1-score voor iedere mogelijke beslisdrempel."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    # Het laatste punt van de curve hoort niet bij een drempel
    f1_values = (2 * precision[:-1] * recall[:-1]
                 / np.maximum(precision[:-1] + recall[:-1], 1e-12))
    return thresholds, f1_values


def find_best_threshold(y_true, scores):
    """Geeft de drempel met de hoogste F1-score en die F1-score."""
    thresholds, f1_values = f1_per_threshold(y_true, scores)
    best = np.argmax(f1_values)
    return thresholds[best], f1_values[best]


def f1_at_best_threshold(y_true, scores):
    """Scorefunctie voor GridSearchCV: F1 bij de beste drempel."""
    return find_best_threshold(y_true, scores)[1]


F1_BEST_SCORER = make_scorer(
    f1_at_best_threshold,
    response_method=("predict_proba", "decision_function"))


def metrics_from_confusion(tn, fp, fn, tp):
    """Berekent de metrics zelf uit de vier cellen van de matrix."""
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    specificity = tn / (tn + fp) if tn + fp > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)
    return {"accuracy": accuracy, "precision": precision,
            "recall": recall, "specificity": specificity, "f1": f1}


def evaluate_predictions(y_true, y_pred, scores):
    """Combineert de metrics uit de confusion matrix met ROC- en PR-AUC."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    results = metrics_from_confusion(tn, fp, fn, tp)
    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"] = average_precision_score(y_true, scores)
    results.update({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
    return results

<br>

### 2.5 Klassen `ModelExperiment` en `ResultsTable`
Ieder model doorloopt **dezelfde vier stappen**. Die stappen zitten in de klasse `ModelExperiment`, zodat alle modellen op precies dezelfde manier worden getraind en beoordeeld:

1. `tune()` – hyperparameters zoeken met `GridSearchCV` en 5-voudige gestratificeerde cross-validation op de trainingsset.
2. `tune_threshold()` – met *out-of-fold* voorspellingen de beslisdrempel kiezen met de hoogste F1.
3. `evaluate()` – het getunede model testen op de validatieset, die tijdens stap 1 en 2 niet is gebruikt.
4. `make_submission()` – het model opnieuw trainen op **alle** trainingsdata en een csv-bestand maken voor Kaggle.

`ResultsTable` verzamelt de resultaten van alle modellen voor de eindtabel in hoofdstuk 6.

In [5]:
# Prompt 1: <CHAT-TITEL>
class ModelExperiment:
    """Voert de vaste werkwijze (tunen, drempel, evaluatie) uit.

    Parameters
    ----------
    name : str
        Naam van het model in tabellen en bestandsnamen.
    pipeline : Pipeline
        Pipeline met eventueel een scaler, een sampler en het model.
    param_grid : dict or list of dict
        Hyperparameters die GridSearchCV afzoekt.
    techniques : str
        Toegepaste technieken tegen de imbalance (voor de eindtabel).
    """

    def __init__(self, name, pipeline, param_grid, techniques):
        self.name = name
        self.pipeline = pipeline
        self.param_grid = param_grid
        self.techniques = techniques
        self.cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                  random_state=RANDOM_STATE)
        self.search = None
        self.tuning_seconds = None
        self.threshold = None
        self.oof_f1 = None
        self.oof_scores = None
        self.oof_target = None
        self.val_scores = None
        self.val_pred = None
        self.val_metrics = {}
        self.submission_path = None
        self.test_positive_rate = None

    @property
    def best_model(self):
        """Het model met de beste hyperparameters (getraind op train)."""
        return self.search.best_estimator_

    @property
    def uses_probabilities(self):
        """True als het model kansen geeft (predict_proba)."""
        return hasattr(self.best_model, "predict_proba")

    def tune(self, features, target):
        """Stap 1: zoekt de beste hyperparameters met GridSearchCV."""
        self.search = GridSearchCV(self.pipeline, self.param_grid,
                                   scoring=F1_BEST_SCORER, cv=self.cv,
                                   n_jobs=N_JOBS)
        start = time.perf_counter()
        self.search.fit(features, target)
        self.tuning_seconds = time.perf_counter() - start
        return self

    def tune_threshold(self, features, target):
        """Stap 2: kiest de drempel met de hoogste out-of-fold F1."""
        method = ("predict_proba" if self.uses_probabilities
                  else "decision_function")
        scores = cross_val_predict(clone(self.best_model), features, target,
                                   cv=self.cv, method=method, n_jobs=N_JOBS)
        self.oof_scores = scores[:, 1] if scores.ndim == 2 else scores
        self.oof_target = np.asarray(target)
        self.threshold, self.oof_f1 = find_best_threshold(
            self.oof_target, self.oof_scores)
        return self

    def evaluate(self, features, target):
        """Stap 3: test het model op de validatieset."""
        self.val_scores = get_scores(self.best_model, features)
        self.val_pred = (self.val_scores >= self.threshold).astype(int)
        self.val_metrics = evaluate_predictions(target, self.val_pred,
                                                self.val_scores)
        # Ter vergelijking: F1 met de standaarddrempel (0,5 of 0)
        default = 0.5 if self.uses_probabilities else 0.0
        self.val_metrics["f1_default"] = f1_score(
            target, (self.val_scores >= default).astype(int))
        return self

    def make_submission(self, features, target, test_features, test_ids):
        """Stap 4: traint op alle data en schrijft een Kaggle-bestand."""
        final_model = clone(self.best_model).fit(features, target)
        scores = get_scores(final_model, test_features)
        predictions = (scores >= self.threshold).astype(int)
        file_name = ("submission_" + self.name.lower().replace(" ", "_")
                     + ".csv")
        self.submission_path = SUBMISSION_DIR / file_name
        submission = pd.DataFrame({"id": test_ids, "stroke": predictions})
        submission.to_csv(self.submission_path, index=False)
        self.test_positive_rate = predictions.mean()
        return self

    def run(self, data):
        """Voert stap 1 t/m 4 uit; data is een dict met alle splits."""
        self.tune(data["X_train"], data["y_train"])
        self.tune_threshold(data["X_train"], data["y_train"])
        self.evaluate(data["X_val"], data["y_val"])
        self.make_submission(data["X_full"], data["y_full"],
                             data["X_test"], data["test_ids"])
        self.print_summary()
        return self

    def best_params_text(self):
        """Geeft de beste hyperparameters als korte tekst."""
        return ", ".join(f"{key.split('__')[-1]}={value}"
                         for key, value in self.search.best_params_.items())

    def print_summary(self):
        """Print een korte samenvatting van de resultaten."""
        metrics = self.val_metrics
        print(f"Model: {self.name}")
        print(f"Beste hyperparameters: {self.best_params_text()}")
        print(f"CV-F1 (5 folds, beste drempel per fold): "
              f"{self.search.best_score_:.3f}")
        print(f"Gekozen drempel (out-of-fold): {self.threshold:.3f}")
        print(f"Validatie-F1: {metrics['f1']:.3f} met gekozen drempel, "
              f"{metrics['f1_default']:.3f} met standaarddrempel")
        print(f"Validatie precision: {metrics['precision']:.3f}, "
              f"recall: {metrics['recall']:.3f}")
        print(f"Tuningtijd: {self.tuning_seconds:.0f} s | "
              f"submission: {self.submission_path}")

    def plot_evaluation(self, y_val):
        """Toont de confusion matrix op de validatieset."""
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_val, self.val_pred, display_labels=["Geen stroke", "Stroke"],
            cmap="Blues", colorbar=False, ax=ax)
        ax.set(title=f"{self.name}: validatieset\n"
                     f"(drempel = {self.threshold:.3f})",
               xlabel="Voorspeld", ylabel="Werkelijk")
        ax.grid(False)
        plt.tight_layout()
        plt.show()

    def summary(self):
        """Geeft de resultaten als dict voor de eindtabel."""
        metrics = self.val_metrics
        return {"model": self.name,
                "technieken imbalance": self.techniques,
                "CV-F1": self.search.best_score_,
                "validatie-F1": metrics["f1"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "ROC-AUC": metrics["roc_auc"],
                "PR-AUC": metrics["pr_auc"],
                "drempel": self.threshold,
                "beste hyperparameters": self.best_params_text()}


class ResultsTable:
    """Verzamelt de resultaten van alle experimenten in één tabel."""

    def __init__(self):
        self.experiments = {}

    def add(self, experiment):
        """Voegt een uitgevoerd experiment toe."""
        self.experiments[experiment.name] = experiment

    def to_frame(self, kaggle_scores=None):
        """Maakt een DataFrame met één rij per model."""
        rows = [experiment.summary()
                for experiment in self.experiments.values()]
        table = pd.DataFrame(rows).set_index("model")
        if kaggle_scores is not None:
            table.insert(2, "Kaggle-F1", pd.Series(kaggle_scores))
        return table

### Data inladen

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




vcguhctg